# Lab 3 — AlexNet / VGG16 / ResNet50 从头训练（Colab 版）

## 开 GPU 的两步

1. 顶部菜单 **「代码执行程序」** → **「更改运行时类型」**
2. **硬件加速器** 选 **T4 GPU** → 保存

然后依次运行下面的单元。

> 作业要求：三个模型都**不能用预训练权重**，全部从零训练。
> 代码里用 `weights=None` 保证这一点。


## 1. 检查 GPU

In [ ]:
import sys, os, time, json, csv, io, random
import torch, torchvision

print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu        :", torch.cuda.get_device_name(0))
else:
    print("\n*** 没有 GPU！请按上面的两步开启 T4，再重新运行本单元 ***")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using:", DEVICE)


## 2. 准备数据

ImageNette 是从 ImageNet 抽出的 10 个类别，作业允许使用缩减版 ImageNet：
> the student may use a reduced version of the ImageNet dataset


In [ ]:
from pathlib import Path
import urllib.request

BASE = Path("/content/lab3")
PROC = BASE / "processed"
BASE.mkdir(parents=True, exist_ok=True)
PARQUET = BASE / "imagenette.parquet"

URL = ("https://huggingface.co/datasets/johnowhitaker/imagenette2-320"
       "/resolve/main/data/train-00000-of-00001.parquet")

if not PARQUET.exists():
    print("downloading (332 MB) ...")
    t0 = time.time()
    urllib.request.urlretrieve(URL, PARQUET)
    print(f"done in {time.time()-t0:.0f}s")
print("size:", round(PARQUET.stat().st_size/1024/1024, 1), "MB")


In [ ]:
import pandas as pd
from PIL import Image

VAL_PER_CLASS = 150
SEED = 42
CLASSES = ["tench", "English springer", "cassette player", "chain saw", "church",
           "French horn", "garbage truck", "gas pump", "golf ball", "parachute"]

if not (PROC / "split.json").exists():
    df = pd.read_parquet(PARQUET)
    print("rows:", len(df))
    rng = random.Random(SEED)
    for split in ("train", "val"):
        (PROC / split).mkdir(parents=True, exist_ok=True)

    counts = {"train": 0, "val": 0}
    for label in sorted(df["label"].unique()):
        sub = df[df["label"] == label]
        idx = list(range(len(sub)))
        rng.shuffle(idx)
        folder = f"{label:02d}_{CLASSES[label].replace(' ', '_')}"
        for split in ("train", "val"):
            (PROC / split / folder).mkdir(exist_ok=True)
        for pos, ri in enumerate(idx):
            raw = sub.iloc[ri]["image"]["bytes"]
            split = "val" if pos < VAL_PER_CLASS else "train"
            try:
                im = Image.open(io.BytesIO(raw)).convert("RGB")
            except Exception:
                continue
            im.save(PROC / split / folder / f"{pos:05d}.jpg", quality=92)
            counts[split] += 1
    (PROC / "split.json").write_text(json.dumps(
        {"classes": CLASSES, "counts": counts}), encoding="utf-8")
    print(counts)

for split in ("train", "val"):
    print(f"{split}: {sum(1 for _ in (PROC/split).rglob('*.jpg'))} images")


## 3. 测试集（50 张自建图）

两种方式，任选：

**方式 A（推荐）**：把 `testset` 文件夹上传到 Google Drive 的 `MyDrive/lab3/testset`，
下面第一个单元会把 Drive 挂上来。

**方式 B**：跳过 Drive，把 `testset` 直接传到 `/content/lab3/testset`。


In [ ]:
# 从 GitHub 拉取 50 张测试图
# 把下面两行改成你自己的用户名与仓库名
GH_USER = "Ouyangjunzhe"
GH_REPO = "lab3"
GH_BRANCH = "main"

import urllib.request, json, io, csv
from pathlib import Path

DEST = Path("/content/lab3/testset")

README_URL = (f"https://raw.githubusercontent.com/"
              f"{GH_USER}/{GH_REPO}/{GH_BRANCH}/README.md")

def _reachable() -> bool:
    try:
        urllib.request.urlopen(README_URL, timeout=10).read()
        return True
    except Exception:
        return False

if DEST.exists() and (DEST / "labels.csv").exists():
    print("test set already present:", DEST)
elif _reachable():
    base = (f"https://raw.githubusercontent.com/"
            f"{GH_USER}/{GH_REPO}/{GH_BRANCH}/testset")
    url = base + "/labels.csv"
    print("fetching labels from", url)
    rows = list(csv.DictReader(io.StringIO(
        urllib.request.urlopen(url, timeout=30).read().decode("utf-8-sig"))))
    DEST.mkdir(parents=True, exist_ok=True)
    for i, r in enumerate(rows, 1):
        u = f"{base}/{r['file']}"
        data = urllib.request.urlopen(u, timeout=30).read()
        dst = DEST / r["file"]
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_bytes(data)
        if i % 10 == 0:
            print(f"  {i}/{len(rows)}")
    with open(DEST / "labels.csv", "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print("done:", len(rows), "images")
else:
    print("GitHub 仓库不可达 —— 请检查 GH_USER / GH_REPO 是否填对，")
    print("或者把 testset 文件夹手动传到 /content/lab3/testset")

n = sum(1 for _ in DEST.rglob("*.jpg")) if DEST.exists() else 0
print(f"{DEST}: {n} images")


In [ ]:
# 方式 A：挂载 Drive（若已把 testset 放进 MyDrive/lab3/ 则能自动找到）
USE_DRIVE = False      # 若要把测试集放 Drive，改成 True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("drive mounted")

TEST_CANDIDATES = [
    Path("/content/drive/MyDrive/lab3/testset"),
    BASE / "testset",
    Path("/content/testset"),
]
TEST = next((p for p in TEST_CANDIDATES if (p / "labels.csv").exists()), None)
if TEST:
    print(f"test set found: {TEST}  "
          f"({sum(1 for _ in TEST.rglob('*.jpg'))} images)")
else:
    print("test set NOT found — will evaluate on validation set instead")
    print("把 testset 文件夹传到 /content/lab3/testset 即可")


## 4. 模型定义（不使用预训练权重）

In [ ]:
import torch.nn as nn
from torchvision import models

NUM_CLASSES = 10
INPUT_SIZE = 128      # 标准 224，这里降分辨率加快训练

def build_model(name):
    # weights=None：明确不使用任何预训练权重
    if name == "AlexNet":
        return models.alexnet(weights=None, num_classes=NUM_CLASSES)
    if name == "VGG16":
        return models.vgg16(weights=None, num_classes=NUM_CLASSES)
    if name == "ResNet50":
        return models.resnet50(weights=None, num_classes=NUM_CLASSES)
    raise ValueError(name)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def depth(m):
    return sum(1 for x in m.modules() if isinstance(x, (nn.Conv2d, nn.Linear)))

for name in ("AlexNet", "VGG16", "ResNet50"):
    m = build_model(name)
    print(f"{name:<10} params={n_params(m):>12,}  depth={depth(m)}")
    del m
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 5. 训练

三个模型用完全相同的超参数，保证对比公平。


In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

BATCH = 64
EPOCHS = 20
LR = 0.01
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize(int(INPUT_SIZE*1.14)),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(PROC/"train", train_tf)
val_ds   = datasets.ImageFolder(PROC/"val", val_tf)
train_dl = DataLoader(train_ds, BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(train_ds)}  val {len(val_ds)}  batches/epoch {len(train_dl)}")


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    c1 = c5 = tot = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        _, t5 = out.topk(5, dim=1)
        c5 += (t5 == y.unsqueeze(1)).any(1).sum().item()
        c1 += (out.argmax(1) == y).sum().item()
        tot += y.size(0)
    return c1/tot, c5/tot

def train_model(name, epochs=EPOCHS):
    torch.manual_seed(42)
    model = build_model(name).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9,
                          weight_decay=5e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    hist = []
    t0 = time.time()
    for ep in range(1, epochs+1):
        model.train()
        run = seen = 0
        for i, (x, y) in enumerate(train_dl, 1):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            run += loss.item()*y.size(0); seen += y.size(0)
        sch.step()
        t1, t5 = evaluate(model, val_dl)
        dt = time.time()-t0
        hist.append({"epoch": ep, "loss": run/seen,
                     "val_top1": t1, "val_top5": t5})
        print(f"{name} ep {ep:>2}/{epochs}  loss {run/seen:.3f}  "
              f"val top1 {t1*100:5.2f}%  top5 {t5*100:5.2f}%  [{dt/60:.1f} min]")

    torch.save(model.state_dict(), f"/content/lab3/{name}.pth")
    return model, hist


In [ ]:
def train_one(name, epochs=EPOCHS, force=False):
    """
    训练单个模型并立刻存盘。

    若 /content/lab3/{name}.pth 已存在且 force=False，直接加载后返回，
    这样断线重连后重跑本单元不会重训已经完成的模型。
    """
    import os
    weight = f"/content/lab3/{name}.pth"
    hist_path = f"/content/lab3/{name}_history.json"

    torch.manual_seed(42)
    model = build_model(name).to(DEVICE)

    if os.path.exists(weight) and not force:
        print(f"[{name}] found {weight}, loading instead of training")
        model.load_state_dict(torch.load(weight, map_location=DEVICE))
        hist = json.load(open(hist_path)) if os.path.exists(hist_path) else []
        return model, hist

    npar, dep = n_params(model), depth(model)
    print(f"[{name}] params {npar:,}  depth {dep}")

    crit = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9,
                          weight_decay=5e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    hist = []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        run = seen = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            run += loss.item() * y.size(0)
            seen += y.size(0)
        sch.step()
        t1, t5 = evaluate(model, val_dl)
        dt = time.time() - t0
        hist.append({"epoch": ep, "loss": run / seen,
                     "val_top1": t1, "val_top5": t5})
        print(f"  {name} ep {ep:>2}/{epochs}  loss {run/seen:.3f}  "
              f"val top1 {t1*100:5.2f}%  top5 {t5*100:5.2f}%  [{dt/60:.1f} min]")

    # 立刻存盘，断线也不丢
    torch.save(model.state_dict(), weight)
    json.dump(hist, open(hist_path, "w"))
    print(f"  saved -> {weight}")
    return model, hist


def register(name, model, hist):
    """把结果登记到全局字典，供后面的评估与画图使用。"""
    results[name] = {"history": hist,
                     "parameters": n_params(model),
                     "depth": depth(model)}
    models_trained[name] = model


第一段：AlexNet

训练完会自动存到 `/content/lab3/AlexNet.pth`。
断开后重跑本单元会跳过训练，直接加载已有权重。

In [ ]:
_m, _h = train_one("AlexNet")
register("AlexNet", _m, _h)
print("AlexNet ready")

第二段：VGG16

参数量最大（1.34 亿）。如果显存不足报错，把上面的 `BATCH` 从 64 改成 32。

In [ ]:
_m, _h = train_one("VGG16")
register("VGG16", _m, _h)
print("VGG16 ready")

第三段：ResNet50

残差网络，同样轮数下通常比 VGG16 略快。

In [ ]:
_m, _h = train_one("ResNet50")
register("ResNet50", _m, _h)
print("ResNet50 ready")

## 6. 在 50 张测试集上评估

In [ ]:
class TestSet(Dataset):
    def __init__(self, root, tf):
        self.root, self.tf = root, tf
        with open(root/"labels.csv", encoding="utf-8-sig") as fh:
            self.rows = list(csv.DictReader(fh))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(self.root/r["file"]).convert("RGB")
        return self.tf(img), int(r["class_index"]), r["file"]

eval_dl = DataLoader(TestSet(TEST, val_tf), 32) if TEST else val_dl
print("evaluating on:", "test set" if TEST else "validation set")

summary = []
for name, m in models_trained.items():
    t0 = time.time()
    t1, t5 = evaluate(m, eval_dl)
    secs = time.time()-t0
    summary.append({"model": name, "top1": t1, "top5": t5,
                    "parameters": results[name]["parameters"],
                    "depth": results[name]["depth"], "seconds": secs})
    print(f"{name:<10} top1 {t1*100:5.1f}%  top5 {t5*100:5.1f}%  "
          f"params {results[name]['parameters']:>12,}  "
          f"depth {results[name]['depth']:>3}  {secs:.2f}s")


In [ ]:
import pandas as pd
comp = pd.DataFrame([{
    "Model": r["model"],
    "Top-1 accuracy": f"{r['top1']*100:.1f}%",
    "Top-5 accuracy": f"{r['top5']*100:.1f}%",
    "Parameters": f"{r['parameters']:,}",
    "Depth": r["depth"],
    "Time for 50 images (s)": round(r["seconds"], 2),
} for r in summary])
comp.to_csv("/content/lab3/comparison.csv", index=False)
comp


## 7. 训练曲线

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, r in results.items():
    eps = [h["epoch"] for h in r["history"]]
    axes[0].plot(eps, [h["loss"] for h in r["history"]], "o-", label=name)
    axes[1].plot(eps, [h["val_top1"]*100 for h in r["history"]], "o-", label=name)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("training loss")
axes[0].set_title("Training loss"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation top-1 (%)")
axes[1].set_title("Validation top-1"); axes[1].legend(); axes[1].grid(alpha=.3)
fig.tight_layout()
fig.savefig("/content/lab3/training_curves.png", dpi=130)
plt.show()


## 8. 打包下载

In [ ]:
import shutil
json.dump({"results": results, "eval": summary},
          open("/content/lab3/results.json","w"), indent=2)

shutil.make_archive("/content/lab3_results", "zip", "/content/lab3",
                    base_dir=None)
print("created /content/lab3_results.zip")

from google.colab import files
files.download("/content/lab3_results.zip")
